# 03 - Model training

Train the baseline CNN first, then transfer-learning models, and compare them on the **validation** set. Do not look at the test set here.

**Running on Google Colab?** Upload or clone the project, then run once:

```python
from google.colab import drive
drive.mount("/content/drive")          # optional: keep data and models on Drive
%cd /content/iClaim
%pip install -q -r requirements.txt
```
Use a GPU runtime (Runtime > Change runtime type) for training.

In [ ]:
# --- Setup (works locally and in Google Colab) ---
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.config import load_config, resolve_path, set_seed
cfg = load_config()
set_seed(cfg["project"]["seed"])
print("Project root:", ROOT)

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 1. Baseline CNN

Sets a performance floor. Reduce epochs for a quick first run.

In [ ]:
from src.train import train

baseline_path = train(cfg, backbone="baseline", output="models/baseline.keras", epochs_head=15, epochs_finetune=0)

## 2. Transfer learning (MobileNetV2)

Phase 1 trains the new head; phase 2 fine-tunes the top backbone layers at a low learning rate.

In [ ]:
mobilenet_path = train(cfg, backbone="mobilenetv2", output="models/mobilenetv2.keras")

## 3. Other backbones (optional)

Uncomment to compare. Keep the model with the best validation macro-F1 / loss.

In [ ]:
# resnet_path = train(cfg, backbone="resnet50", output="models/resnet50.keras")
# effnet_path = train(cfg, backbone="efficientnetb0", output="models/efficientnetb0.keras")

## 4. Training curves

Look for overfitting: validation loss rising while training loss keeps falling.

In [ ]:
import json
for name in ["baseline", "mobilenetv2"]:
    f = resolve_path(cfg["paths"]["figures_dir"]) / f"history_{name}.json"
    if f.exists():
        h = json.loads(f.read_text())
        print(name, "best val_accuracy: %.4f" % max(h["val_accuracy"]), "| lowest val_loss: %.4f" % min(h["val_loss"]))

## 5. Pick the final model

Copy the chosen model to `models/best_model.keras` (the file the Streamlit app loads):

In [ ]:
import shutil
chosen = "models/mobilenetv2.keras"   # change after comparing validation results
shutil.copy(resolve_path(chosen), resolve_path(cfg["paths"]["best_model"]))
print("best_model.keras updated from", chosen)